In [1]:
import os
import certifi
from dotenv import load_dotenv
from IPython.display import display, HTML
import requests

from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_community.tools.tavily_search import TavilySearchResults #Used to access prompts stored in LangChain Hub.
from langchain import hub # provides reusable prompts


In [2]:
#Agent related functionalities
#Used to create and run an AI agent that can decide when to use tools.
from langchain.agents import create_react_agent, AgentExecutor

In [3]:

from langchain_core.tools import tool
## @tool turns a normal Python function into a tool that an AI agent can understand and use.
@tool
def add(a: int, b: int) -> int:
    """Add two numbers together"""
    return a + b
#Now function add becomes a LangChain Tool.

In [4]:
#HTTPS = a secure connection between your Python program and a website/API.
#Certificate = a digital ID that helps verify that the website/API is genuine.
#certifi = provides a collection of trusted certificates.
#certifi.where() = finds where that certificate file is stored.
#os.environ["SSL_CERT_FILE"] = tells Python to use that certificate file.

#os.environ["SSL_CERT_FILE"] = certifi.where()

#Load ENV Variables
load_dotenv()

OPEN_API_KEY = os.getenv("OPEN_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [5]:
#Initializing LLM
#llm = ChatOpenAI(model="gpt-6-luna", temperature=0, openai_api_key=OPEN_API_KEY)
llm = ChatOllama(model="llama3.2:latest", temperature=0)
#ollama lis

In [6]:
from datetime import datetime
from langchain_core.tools import tool

@tool
def get_current_date(query: str="") -> str:
    """ONLY use this tool when the user asks for today's date,
    current date, or today's day/date."""
    return datetime.now().strftime("%d-%b-%Y")

In [7]:
@tool
def get_weather_data(location: str) -> str:
    """ONLY use this tool when the user asks for current weather,
    temperature, humidity, or weather conditions for a specific city.
    The location should be a city name such as New Delhi, London, or Chicago."""
    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={location}"
    )
    response = requests.get(url)
    data = response.json()
    
    if "current" not in data:
        return f"Could not fetch weather data for {location}"
    return (
        f"City: {location}\n"
        f"Temperature: {data['current']['temperature']}deg C \n"
        f"Weather: {data['current']['weather_descriptions'][0]} \n"
        f"Humidity: {data['current']['humidity']}%"
    )
    
llm_with_weather_tool = llm.bind_tools([get_weather_data])
response = llm_with_weather_tool.invoke("What is the current weather in New Delhi?")
print(response.tool_calls)

[{'name': 'get_weather_data', 'args': {'location': 'New Delhi'}, 'id': 'f1d7e3ac-dc1c-4f97-8836-b4568ad959a2', 'type': 'tool_call'}]


In [8]:
llm_with_date_tool = llm.bind_tools([get_current_date])
response = llm_with_date_tool.invoke("What is today's date")
if response.tool_calls:
    date = get_current_date.invoke(response.tool_calls[0]["args"])
    print(date)


04-Oct-2026


In [9]:
#Initializing Tavily Search Result Object
search_tool = TavilySearchResults(max_results=2)
result = search_tool.invoke("Give me the latest news on AI")
result

[{'url': 'https://www.artificialintelligence-news.com',
  'content': 'Retail & Logistics AI\n\nSeptember 18, 2026\n\n### ChatGPT pioneer launches Jev model for programmatic logic\n\nAI Startups & Funding\n\nSeptember 16, 2026\n\n### MIT AI forecasts extreme weather without historical data\n\nEnvironment & Sustainability\n\nAugust 25, 2026\n\n[](\n\n#### Enterprise\n\n### Malaysia’s MACC expands AI use for intelligence-led investigations\n\nAI in Action\n\nOctober 2, 2026\n\n### Meta Muse gains enterprise software connectors\n\nWorld of Work\n\nSeptember 29, 2026\n\n### AI weather forecasting enters the energy market as Google targets grid operators with WeatherNext 3\n\nArtificial Intelligence\n\nSeptember 8, 2026\n\n#### Industries\n\n### How McDonald’s uses AI to recommend menu prices\n\nAI in Action\n\nSeptember 30, 2026\n\n### Meta Muse gains enterprise software connectors\n\nWorld of Work [...] October 2, 2026\n\nMalaysia’s MACC expands AI use for intelligence-led investigations\n

In [10]:
llm_with_tavily_tool = llm.bind_tools([search_tool])

response = llm_with_tavily_tool.invoke("What is the top latest news on AI?")

if response.tool_calls:
    result = search_tool.invoke(response.tool_calls[0]["args"])

    for item in result:
        print(item)

        if "url" in item:
            display(
                HTML(
                    f'<a href="{item["url"]}" target="_blank">{item["url"]}</a>'
                )
            )

#search_tool = the tool
#llm_with_tavily_tool = LLM with that tool attached.

{'url': 'https://www.artificialintelligence-news.com', 'content': 'Retail & Logistics AI\n\nSeptember 18, 2026\n\n### ChatGPT pioneer launches Jev model for programmatic logic\n\nAI Startups & Funding\n\nSeptember 16, 2026\n\n### MIT AI forecasts extreme weather without historical data\n\nEnvironment & Sustainability\n\nAugust 25, 2026\n\n[](\n\n#### Enterprise\n\n### Malaysia’s MACC expands AI use for intelligence-led investigations\n\nAI in Action\n\nOctober 2, 2026\n\n### Meta Muse gains enterprise software connectors\n\nWorld of Work\n\nSeptember 29, 2026\n\n### AI weather forecasting enters the energy market as Google targets grid operators with WeatherNext 3\n\nArtificial Intelligence\n\nSeptember 8, 2026\n\n#### Industries\n\n### How McDonald’s uses AI to recommend menu prices\n\nAI in Action\n\nSeptember 30, 2026\n\n### Meta Muse gains enterprise software connectors\n\nWorld of Work [...] October 2, 2026\n\nMalaysia’s MACC expands AI use for intelligence-led investigations\n\nA

{'url': 'https://www.wsj.com/tech/ai?eafs_enabled=false', 'content': "4\n\nThe smart-ring maker said it was delaying its listing despite strong demand.\n\n172\n\nRisk and Compliance Journal\n\nPlus: U.S. extends 50% rule start under China trade truce, and the possible next House intel chair aims to probe ‘nakedly corrupt dealings’\n\nGoogle said it has filed appeals with the European Union’s second-highest court against binding instructions to open up its Android operating system, allowing other search engines to access search data.\n\nThe 10-Point\n\nPlus, Middle East oil exports rebound and Goldman’s board discusses a plan to name its next CEO.\n\nThe investment in Helix Digital Infrastructure brings the platform’s total secured capital to more than $11 billion. [...] Stocks\n\nMarket choppiness and AI safety fears are spooking investors and throwing into disarray what was expected to be a string of major IPOs this fall.\n\n76\n\nThe more than $20 billion award for F/A-XX adds to str

In [11]:
print(response.tool_calls)

[{'name': 'tavily_search_results_json', 'args': {'query': 'latest news AI'}, 'id': 'ab781557-9136-4ad4-8120-d4acf54d1158', 'type': 'tool_call'}]


In [12]:
import yfinance as yf

@tool
def get_top_5_gainers(query: str="") -> str:
    """ONLY use this tool when the user asks for stock market gainers,
    top gaining stocks, biggest stock winners, or today's stock gainers.
    Do NOT use this tool for weather, dates, cities, countries, or general questions."""

    data = yf.screen("day_gainers")

    top_5 = data["quotes"][:5]

    return [
        {
            "symbol": stock["symbol"],
            "name": stock["shortName"],
            "price": stock["regularMarketPrice"],
            "change_percent": stock["regularMarketChangePercent"]
        }
        for stock in top_5
    ]


llm_with_stock_tool = llm.bind_tools([get_top_5_gainers])
response = llm_with_stock_tool.invoke("What are the top 5 gainers in the US stock market today?")
if response.tool_calls:
    
    result = get_top_5_gainers.invoke(
        response.tool_calls[0]["args"]
    )

    for stock in result:
        print(
            stock["symbol"],
            "-",
            stock["name"],
            "- $",
            stock["price"],
            "-",
            round(stock["change_percent"], 2),
            "%"
        )



MXL - MaxLinear, Inc - $ 105.93 - 14.99 %
IMOS - ChipMOS TECHNOLOGIES INC. - $ 83.3 - 14.61 %
SYNA - Synaptics Incorporated - $ 121.1 - 14.08 %
PENG - Penguin Solutions, Inc. - $ 61.36 - 11.6 %
VSH - Vishay Intertechnology, Inc. - $ 38.18 - 11.41 %


In [13]:
tools = [search_tool, get_current_date, get_top_5_gainers, get_weather_data]

In [14]:
#Prompt
#For create react agent functionality, below is recommended prompt
prompt1 = hub.pull("hwchase17/react")

In [15]:
from langchain_core.prompts import PromptTemplate

template = """Answer the question using the available tool when necessary.

TOOLS:
{tools}

AVAILABLE TOOL NAMES:
{tool_names}

STRICT RULES:

1. The Action field MUST contain ONLY a tool name.
2. The tool name MUST be exactly one of [{tool_names}].
3. Action Input MUST contain only the input for that tool.
4. Use the tool AT MOST ONCE.
5. After you receive an Observation, DO NOT call any tool again.
6. After the Observation, immediately produce:
   Thought: I now know the final answer
   Final Answer: <answer>
7. Never repeat the Question.
8. Never repeat the Observation.
9. Never repeat the search.
10. Final Answer must directly answer the original question.

EXACT FORMAT:

Question: {input}
Thought: I need to find the answer.
Action: <exact tool name>
Action Input: <tool input>
Observation: <tool result>
Thought: I now know the final answer
Final Answer: <answer> the final answer tothe original question

Begin!

Question: {input}
Thought: {agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)

In [16]:
from langchain_core.prompts import PromptTemplate

template = """Answer the question using the available tools when necessary.

TOOLS:
{tools}

AVAILABLE TOOL NAMES:
{tool_names}

Follow this format:

Question: the user's question
Thought: think about what to do
Action: one of [{tool_names}]
Action Input: input for the selected tool
Observation: tool result
Thought: I now know the final answer
Final Answer: the answer to the user's question

RULES:
- Action must contain ONLY the exact tool name.
- Action Input must contain only the tool input.
- Choose only the tool that is relevant to the question.
- After receiving an Observation that answers the question, do NOT call another tool.
- Give the Final Answer immediately.
- Do not repeat the Question.
- Do not repeat the Observation.

Begin!

Question: {input}
Thought: {agent_scratchpad}"""

prompt2 = PromptTemplate.from_template(template)

In [17]:
prompt3 = PromptTemplate.from_template("""
Answer the user's question using the available tools.

TOOLS:
{tools}

Tool names:
{tool_names}

Use this format:

Question: {input}
Thought: decide what to do
Action: tool name
Action Input: tool input
Observation: tool result

Repeat Thought → Action → Action Input → Observation
if another tool is needed.

Important:
- Complete every part of the user's question.
- Use the result of one tool as input to another tool when needed.
- Do not repeat a tool call if its result already answers that part.
- Give Final Answer only after all requested tasks are complete.

When finished:
Thought: I now know the final answer
Final Answer: your complete answer

Question: {input}
Thought: {agent_scratchpad}
""")


In [18]:
search_tool.name

'tavily_search_results_json'

In [19]:
response = llm_with_tavily_tool.invoke(
    "Search for the capital of India."
)

print(response.tool_calls)

[{'name': 'tavily_search_results_json', 'args': {'query': 'capital of India'}, 'id': 'a05c2fea-4436-444e-b46c-073c0b3c7ddb', 'type': 'tool_call'}]


In [20]:
#Create Agent
agent = create_react_agent(
    llm=llm ,
    tools=tools,
    prompt=prompt1
    )
#agent = agent object

In [21]:
#Executor
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools,
    verbose=True,
    max_iterations=3,
    handle_parsing_errors=True
    )
#agent_executor = agent executor object

In [22]:
#Run
response = agent_executor.invoke({
    "input": (
        "What is the capital of India?"
        )                         
    })
print(response["output"])



> Entering new AgentExecutor chain...
Question: What is the capital of India?
Thought: I should use the tavily_search_results_json tool to find the answer, as it is a reliable source for comprehensive and accurate information about current events.

Action: tavily_search_results_json
Action Input: "What is the capital of India?"[{'url': 'https://en.wikipedia.org/wiki/List_of_capitals_of_India', 'content': '## Modern period\n\n[edit]\n\n In 1858, Allahabad the capital of India for a day when it also served as the capital of North-Western Provinces.\n During the British Raj, until 1911, Calcutta was the capital of India.\n By the latter half of the 19th century, Shimla had become the summer capital.\n King George V proclaimed the transfer of the capital from Calcutta to Delhi at the climax of the 1911 Delhi Durbar on 12 December 1911. The buildings housing the Viceroy, government, and parliament were inaugurated in early 1931.\n\n## References\n\n[edit] [...] Appearance\n\nFrom Wikipedi

In [23]:
print([tool.name for tool in agent_executor.tools])

['tavily_search_results_json', 'get_current_date', 'get_top_5_gainers', 'get_weather_data']
